# INCEpTION gold corpus — exploratory data analysis (A)

This notebook measures the five flattened tables in
`data/interim/gcn_gold_corpus/`: what each contains, how the baseline
pre-annotation compares to what annotators produced, and where the two
disagree. `layer_source` is either `INITIAL_CAS` — the pre-annotation
baseline every annotator started from — or an annotator's username.

It is **descriptive only**. It cleans nothing, decides nothing, and
writes no data file. Decisions taken from these measurements appear in
the final cell.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/interim/gcn_gold_corpus").is_dir())
CORPUS = ROOT / "data/interim/gcn_gold_corpus"

TABLE_NAMES = ["documents", "annotators", "evidence_spans", "photometry_spans", "event_summaries"]
EXPECTED_SHAPE = {"documents": (10, 5), "annotators": (28, 10), "evidence_spans": (6610, 12),
                  "photometry_spans": (2741, 22), "event_summaries": (28, 26)}

TABLES = {}
for name in TABLE_NAMES:
    path = CORPUS / f"{name}.parquet"
    frame = pd.read_parquet(path)
    if len(frame) == 0:
        raise ValueError(f"Table '{name}' loaded 0 rows from {path}")
    TABLES[name] = frame
    print(f"{name:18s} rows={len(frame):5d} columns={frame.shape[1]:4d}  <- {path}")

STRUCTURAL_SPAN_COLS = ["document_name", "layer_source", "xmi_id", "begin", "end", "covered_text"]
STRUCTURAL_DOC_COLS = ["document_name", "layer_source", "xmi_id"]
EVIDENCE_FEATURES = [c for c in TABLES["evidence_spans"].columns if c not in STRUCTURAL_SPAN_COLS]
PHOTOMETRY_FEATURES = [c for c in TABLES["photometry_spans"].columns if c not in STRUCTURAL_SPAN_COLS]
EVENT_SUMMARY_FEATURES = [c for c in TABLES["event_summaries"].columns if c not in STRUCTURAL_DOC_COLS]


def clip(value, limit=60):
    """Keep any displayed value short."""
    text = str(value)
    return f"{text[:limit]}..." if len(text) > limit else text


def feature_states(series):
    """(non_null, empty_string, populated): null = attribute absent from the XMI,
    '' = attribute present and empty, anything else = populated content."""
    non_null = int(series.notna().sum())
    empty = int((series == "").sum())
    return non_null, empty, non_null - empty


def summarise_feature(series):
    """One-line description of a feature's populated (non-null, non-empty) values."""
    populated = series[series.notna() & (series != "")]
    if populated.empty:
        return "no populated values"
    distinct = populated.nunique()
    if distinct <= 12:
        return " | ".join(f"{clip(k)}={v}" for k, v in populated.value_counts().items())
    lengths = populated.map(len)
    return f"{distinct} distinct values, length min={lengths.min()} median={lengths.median():.0f} max={lengths.max()}"


def census_table(df, features):
    """One row per feature column: null/empty/populated counts, split baseline vs annotator."""
    baseline = df[df["layer_source"] == "INITIAL_CAS"]
    annotator = df[df["layer_source"] != "INITIAL_CAS"]
    rows = []
    for feature in features:
        b_non_null, b_empty, b_populated = feature_states(baseline[feature])
        a_non_null, a_empty, a_populated = feature_states(annotator[feature])
        rows.append({
            "column": feature,
            "baseline_non_null": b_non_null, "baseline_empty": b_empty, "baseline_populated": b_populated,
            "annotator_non_null": a_non_null, "annotator_empty": a_empty, "annotator_populated": a_populated,
            "distinct": int(df[feature][df[feature].notna() & (df[feature] != "")].nunique()),
            "summary": summarise_feature(df[feature]),
        })
    return pd.DataFrame(rows)


control_rows = [{"control": f"{name} shape", "expected": str(EXPECTED_SHAPE[name]),
                 "observed": str(TABLES[name].shape)} for name in TABLE_NAMES]
initial_cas_docs = {name: TABLES[name].loc[TABLES[name]["layer_source"] == "INITIAL_CAS",
                                            "document_name"].nunique()
                    for name in ["evidence_spans", "photometry_spans"]}
control_rows += [
    {"control": "annotator layers (rows in annotators.parquet)", "expected": "28",
     "observed": str(len(TABLES["annotators"]))},
    {"control": "distinct annotators", "expected": "10",
     "observed": str(TABLES["annotators"]["annotator"].nunique())},
    {"control": "INITIAL_CAS layers in evidence_spans", "expected": "10",
     "observed": str(initial_cas_docs["evidence_spans"])},
    {"control": "INITIAL_CAS layers in photometry_spans", "expected": "10",
     "observed": str(initial_cas_docs["photometry_spans"])},
]
for row in control_rows:
    row["status"] = "PASS" if row["expected"] == row["observed"] else "FAIL"
controls = pd.DataFrame(control_rows)
print("\nCONTROLS")
print(controls.to_string(index=False))

documents          rows=   10 columns=   5  <- /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_corpus/documents.parquet
annotators         rows=   28 columns=  10  <- /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_corpus/annotators.parquet
evidence_spans     rows= 6610 columns=  12  <- /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_corpus/evidence_spans.parquet
photometry_spans   rows= 2741 columns=  22  <- /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_corpus/photometry_spans.parquet
event_summaries    rows=   28 columns=  26  <- /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_corpus/event_summaries.parquet

CONTROLS
                                      control   expected   observed status
                              documents shape    (10, 5)    (10, 5)   PASS
                             annotators shape   (28, 10)   (28, 10)   PASS
                         evidence_spans shape (6610, 12) (661

In [2]:
by_doc = TABLES["annotators"].groupby("document_name")["annotator"]
glance = pd.DataFrame({
    "document": by_doc.nunique().index,
    "annotators": by_doc.nunique().to_numpy(),
    "annotator_names": by_doc.apply(lambda s: ", ".join(sorted(s))).to_numpy(),
}).set_index("document")

baseline_evidence = (TABLES["evidence_spans"].query("layer_source == 'INITIAL_CAS'")
                     .groupby("document_name").size().rename("baseline_evidence"))
baseline_photometry = (TABLES["photometry_spans"].query("layer_source == 'INITIAL_CAS'")
                       .groupby("document_name").size().rename("baseline_photometry"))
text_length = TABLES["documents"].set_index("document_name")["text_length"]

glance = (glance.join(text_length).join(baseline_evidence).join(baseline_photometry)
         [["text_length", "annotators", "annotator_names", "baseline_evidence", "baseline_photometry"]]
         .reset_index().sort_values("document").reset_index(drop=True))
glance

,document,text_length,annotators,annotator_names,baseline_evidence,baseline_photometry
0,event_2025aji.xmi,78950,3,"Camille, Patrice, Xinyue",201,109
1,event_2026owq.xmi,58909,3,"Dahlia, Sarah, Zhanat",204,68
2,event_EP-260623_025405.xmi,26162,2,"Dahlia, Yodgor",96,19
3,event_GCN-251013_173943.xmi,108145,3,"Andrii, Eslam, Sarah",237,266
4,event_GCN-251222_170549.xmi,67663,2,"Camille, Priyadarshini",242,37
5,event_GCN-260604_202037.xmi,56662,2,"Sarah, Yodgor",146,62
6,event_GCN-260614_134953.xmi,16577,2,"Patrice, Priyadarshini",73,7
7,event_GRB-241025_013651.xmi,51059,3,"Camille, Eslam, Xinyue",157,45
8,event_GRB-260708A.xmi,17199,4,"Camille, Eslam, Patrice, Zhanat",68,3
9,event_GRB241030.xmi,79913,4,"Dahlia, Priyadarshini, Yodgor, Zhanat",259,54


In [3]:
evidence_census = census_table(TABLES["evidence_spans"], EVIDENCE_FEATURES)
evidence_census

,column,baseline_non_null,baseline_empty,baseline_populated,annotator_non_null,annotator_empty,annotator_populated,distinct,summary
0,label,1683,0,1683,4909,0,4909,15,"15 distinct values, length min=3 median=14 max=29"
1,target,1680,0,1680,4894,0,4894,5,event=4554 | counterpart=1409 | instrument=565 | nearby_galaxy=24 | host=22
2,certainty,1683,0,1683,4897,0,4897,5,confirmed=5693 | candidate=661 | tentative=178 | rejected=27 | unclear=21
3,value,1683,45,1638,4889,139,4750,454,"454 distinct values, length min=1 median=11 max=58"
4,unit,1683,1516,167,4844,4351,493,25,"25 distinct values, length min=1 median=3 max=40"
5,comment,1683,1353,330,4896,3836,1060,274,"274 distinct values, length min=5 median=26 max=205"


In [4]:
photometry_census = census_table(TABLES["photometry_spans"], PHOTOMETRY_FEATURES)
photometry_census

,column,baseline_non_null,baseline_empty,baseline_populated,annotator_non_null,annotator_empty,annotator_populated,distinct,summary
0,measurement_type,670,0,670,2027,0,2027,2,detection=2129 | upper_limit=568
1,photometric_system,670,0,670,2027,0,2027,3,unknown=1715 | AB=789 | Vega=193
2,target,670,0,670,2069,0,2069,2,counterpart=2530 | event=209
3,certainty,670,0,670,2018,0,2018,3,confirmed=2508 | tentative=174 | candidate=6
4,magnitude_or_limit,670,0,670,2028,0,2028,436,"436 distinct values, length min=1 median=5 max=13"
5,magnitude_error,670,251,419,2017,713,1304,60,"60 distinct values, length min=3 median=4 max=11"
6,limit_sigma,670,661,9,1953,1914,39,7,3=32 | 5=11 | 18.3 mag=1 | 20.9=1 | 10=1 | 0.14=1 | r=1
7,unit,670,0,670,2021,0,2021,1,mag=2691
8,photometric_band,670,48,622,2023,100,1923,98,"98 distinct values, length min=1 median=1 max=10"
9,obs_time_raw,670,87,583,2028,224,1804,598,"598 distinct values, length min=2 median=15 max=67"


In [5]:
event_summary_census = census_table(TABLES["event_summaries"], EVENT_SUMMARY_FEATURES)
print("No INITIAL_CAS rows exist for event_summaries (0 confirmed by the layer_source counts "
     "below), so every baseline_non_null/baseline_empty/baseline_populated value is 0 by "
     "construction -- reported here rather than omitted.")
print(TABLES["event_summaries"]["layer_source"].value_counts().to_string())
event_summary_census

No INITIAL_CAS rows exist for event_summaries (0 confirmed by the layer_source counts below), so every baseline_non_null/baseline_empty/baseline_populated value is 0 by construction -- reported here rather than omitted.
layer_source
Camille          4
Patrice          3
Dahlia           3
Sarah            3
Zhanat           3
Yodgor           3
Priyadarshini    3
Eslam            3
Xinyue           2
Andrii           1


,column,baseline_non_null,baseline_empty,baseline_populated,annotator_non_null,annotator_empty,annotator_populated,distinct,summary
0,canonical_event_name,0,0,0,17,0,17,11,GRB 260708A=3 | GRB 251222A=2 | GRB 250129A=2 | GRB 241030A=2 | GRB 260614B=2 | EP260623a=1 | GRB 260610B=1 | GRB 241025A=1 | GRB 260604C=1 | 260708A=1 | GRB241030A=1
1,main_counterpart_name,0,0,0,11,0,11,11,GRB 250129A=1 | 2026owq=1 | GRB 251222A / EP25122b=1 | EP251222b=1 | AT 2026ofi=1 | SVOM/VT optical counterpart=1 | GRB 241025A / EP241025a=1 | GRB 260708A / AT 2026stf=1 | AT 2026stf=1 | 260708A=1 | Optical afterglow=1
2,best_trigger_time,0,0,0,17,0,17,17,"17 distinct values, length min=11 median=22 max=75"
3,best_trigger_time_source,0,0,0,17,0,17,15,"15 distinct values, length min=5 median=9 max=22"
4,best_redshift_value,0,0,0,10,0,10,7,2.151=2 | 1.411=2 | 3.171=2 | 0.703=1 | 0.473=1 | 4.20=1 | 1.41=1
5,best_redshift_method,0,0,0,10,0,10,1,spectroscopic=10
6,best_redshift_source,0,0,0,10,0,10,9,"37959=2 | VLT=1 | GCN 39071=1 | GCN 45022=1 | 44914=1 | ESO=1 | GCN 43204 (Saccardi et al.)=1 | NOT=1 | Keck/LRIS spectroscopy (GCNs 37959, 38027)=1"
7,best_redshift_comment,0,0,0,4,0,4,4,GTC/OSIRIS+=1 | Spectroscopic redshift confirmed by Saccardi et al. (GCN 432...=1 | Consistent measurements from multiple spectroscopic observat...=1 | data from Keck telescope=1
8,best_t90_value,0,0,0,13,0,13,12,62=2 | 262.25 +- 23.71 sec=1 | 83s=1 | >200s=1 | 6.4=1 | 88 +18/-9 s=1 | 37.6 +1.7/-1.1 s=1 | 62s=1 | 62 s=1 | 208 s=1 | 173.3+/-5.0=1 | 173=1
9,best_t90_instrument,0,0,0,13,0,13,9,Swift BAT=2 | Fermi/GBM=2 | GBM=2 | SVOM/ECLAIRs=2 | FERMI GBM=1 | SVOM GRM=1 | Fermi-GBM=1 | GRBAlpha=1 | Swift/BAT=1


In [6]:
def match_exact(baseline, annotator):
    """Greedy one-to-one match on identical (begin, end).
    Returns (pairs of index labels, unmatched baseline rows, unmatched annotator rows)."""
    by_span = {}
    for idx, row in baseline.iterrows():
        by_span.setdefault((row["begin"], row["end"]), []).append(idx)
    used_b, used_a, pairs = set(), set(), []
    for idx, row in annotator.iterrows():
        for b_idx in by_span.get((row["begin"], row["end"]), []):
            if b_idx not in used_b:
                used_b.add(b_idx); used_a.add(idx); pairs.append((b_idx, idx))
                break
    return pairs, baseline.loc[baseline.index.difference(used_b)], annotator.loc[annotator.index.difference(used_a)]


def match_overlap(baseline, annotator):
    """Greedy one-to-one match by interval overlap, applied only to spans that missed an
    exact match, to measure what a relaxed rule would additionally catch."""
    used_a, pairs = set(), []
    for b_idx, b_row in baseline.iterrows():
        for a_idx, a_row in annotator.iterrows():
            if a_idx not in used_a and b_row["begin"] < a_row["end"] and a_row["begin"] < b_row["end"]:
                used_a.add(a_idx); pairs.append((b_idx, a_idx)); break
    return pairs


PAIRS = list(TABLES["annotators"][["document_name", "annotator"]].itertuples(index=False, name=None))
LAYER_TABLES = {"evidence": TABLES["evidence_spans"], "photometry": TABLES["photometry_spans"]}
FEATURES_BY_LAYER = {"evidence": EVIDENCE_FEATURES, "photometry": PHOTOMETRY_FEATURES}

rows, all_matched_pairs, overlap_examples = [], {"evidence": [], "photometry": []}, []
for document, annotator in PAIRS:
    for layer, table in LAYER_TABLES.items():
        baseline = table[(table["document_name"] == document) & (table["layer_source"] == "INITIAL_CAS")]
        ann = table[(table["document_name"] == document) & (table["layer_source"] == annotator)]
        pairs, b_only, a_only = match_exact(baseline, ann)
        all_matched_pairs[layer] += [(table.loc[b], table.loc[a]) for b, a in pairs]
        rows.append({"document": document, "annotator": annotator, "layer": layer,
                     "baseline_spans": len(baseline), "annotator_spans": len(ann),
                     "matched_exact": len(pairs), "baseline_only": len(b_only), "annotator_only": len(a_only)})
        for b_idx, a_idx in match_overlap(b_only, a_only):
            overlap_examples.append({
                "document": document, "annotator": annotator, "layer": layer,
                "baseline_begin": table.loc[b_idx, "begin"], "baseline_end": table.loc[b_idx, "end"],
                "baseline_text": table.loc[b_idx, "covered_text"],
                "annotator_begin": table.loc[a_idx, "begin"], "annotator_end": table.loc[a_idx, "end"],
                "annotator_text": table.loc[a_idx, "covered_text"]})

match_summary = pd.DataFrame(rows)
print("Totals per layer:")
print(match_summary.groupby("layer")[["baseline_spans", "annotator_spans", "matched_exact",
                                      "baseline_only", "annotator_only"]].sum().to_string())

print(f"\nUnmatched-by-offset pairs that DO overlap "
     f"(exact matching misses, overlap catches): {len(overlap_examples)}")
if overlap_examples:
    print(pd.DataFrame(overlap_examples).head(10).to_string(index=False))
else:
    print("none")
match_summary

Totals per layer:
            baseline_spans  annotator_spans  matched_exact  baseline_only  annotator_only
layer                                                                                    
evidence              4819             4927           4814              5             113
photometry            1942             2071           1942              0             129

Unmatched-by-offset pairs that DO overlap (exact matching misses, overlap catches): 0
none


,document,annotator,layer,baseline_spans,annotator_spans,matched_exact,baseline_only,annotator_only
0,event_2025aji.xmi,Camille,evidence,201,200,197,4,3
1,event_2025aji.xmi,Camille,photometry,109,122,109,0,13
2,event_2025aji.xmi,Patrice,evidence,201,202,201,0,1
3,event_2025aji.xmi,Patrice,photometry,109,109,109,0,0
4,event_2025aji.xmi,Xinyue,evidence,201,202,201,0,1
5,event_2025aji.xmi,Xinyue,photometry,109,109,109,0,0
6,event_2026owq.xmi,Dahlia,evidence,204,204,204,0,0
7,event_2026owq.xmi,Dahlia,photometry,68,69,68,0,1
8,event_2026owq.xmi,Sarah,evidence,204,209,204,0,5
9,event_2026owq.xmi,Sarah,photometry,68,79,68,0,11


In [7]:
diff_rows, diff_pairs_by_feature = [], {}
for layer, features in FEATURES_BY_LAYER.items():
    pairs = all_matched_pairs[layer]
    for feature in features:
        differing = [(b, a) for b, a in pairs if b[feature] != a[feature]]
        diff_pairs_by_feature[(layer, feature)] = differing
        diff_rows.append({"layer": layer, "feature": feature, "matched_pairs": len(pairs),
                          "differing": len(differing),
                          "pct": round(100 * len(differing) / len(pairs), 1) if pairs else 0.0})

feature_diff = pd.DataFrame(diff_rows).sort_values("differing", ascending=False).reset_index(drop=True)

examples = []
for _, row in feature_diff.iterrows():
    pairs = diff_pairs_by_feature[(row["layer"], row["feature"])]
    if pairs:
        b, a = pairs[0]
        examples.append({"layer": row["layer"], "document": b["document_name"], "annotator": a["layer_source"],
                         "feature": row["feature"], "baseline_value": b[row["feature"]],
                         "annotator_value": a[row["feature"]]})
    if len(examples) == 5:
        break

print("Null and an empty string are treated as distinct values here: None != '' compares "
     "unequal, matching the convention the flattening script used to write the tables.")
print("\n5 concrete differing examples, from the features with the most differences:")
print(pd.DataFrame(examples).to_string(index=False))
feature_diff

Null and an empty string are treated as distinct values here: None != '' compares unequal, matching the convention the flattening script used to write the tables.

5 concrete differing examples, from the features with the most differences:
     layer          document annotator            feature                                                                                                                         baseline_value annotator_value
photometry event_2025aji.xmi   Camille            comment           Measurement time not found in the circular; the annotator must confirm it. Photometric system is unknown; verify AB or Vega.            None
photometry event_2025aji.xmi   Camille         instrument                                                                                                                                                    VLT
  evidence event_2025aji.xmi   Camille            comment Trigger time without an adjacent date in the text; the annotator must compl

,layer,feature,matched_pairs,differing,pct
0,photometry,comment,1942,574,29.6
1,photometry,instrument,1942,213,11.0
2,evidence,comment,4814,211,4.4
3,photometry,photometric_system,1942,131,6.7
4,photometry,obs_time_reference,1942,124,6.4
5,photometry,obs_time_raw,1942,69,3.6
6,photometry,exposure_time_raw,1942,39,2.0
7,photometry,photometric_band,1942,34,1.8
8,photometry,obs_time_type,1942,33,1.7
9,evidence,certainty,4814,24,0.5


In [8]:
def comment_class(b_val, a_val):
    b_has = bool(b_val) and b_val.strip() != ""
    a_has = bool(a_val) and a_val.strip() != ""
    if not b_has and not a_has:
        return "both_absent"
    if not b_has and a_has:
        return "baseline_none_annotator_wrote"
    if b_has and not a_has:
        return "annotator_removed"
    return "identical" if b_val == a_val else "annotator_replaced"


comment_rows, replaced_examples = [], []
for layer in ("evidence", "photometry"):
    classes = [comment_class(b["comment"], a["comment"]) for b, a in all_matched_pairs[layer]]
    counts = pd.Series(classes).value_counts()
    comment_rows.append({"layer": layer,
                         "identical": int(counts.get("identical", 0)),
                         "annotator_replaced": int(counts.get("annotator_replaced", 0)),
                         "baseline_none_annotator_wrote": int(counts.get("baseline_none_annotator_wrote", 0)),
                         "annotator_removed": int(counts.get("annotator_removed", 0)),
                         "both_absent": int(counts.get("both_absent", 0))})
    for (b, a), cls in zip(all_matched_pairs[layer], classes):
        if cls == "annotator_replaced" and len(replaced_examples) < 5:
            replaced_examples.append({"layer": layer, "document": b["document_name"],
                                      "annotator": a["layer_source"],
                                      "baseline_comment": b["comment"], "annotator_comment": a["comment"]})

comment_summary = pd.DataFrame(comment_rows)
print("A comment counts as 'present' here only when it is non-null and non-blank; null and an "
     "empty or whitespace-only string are both grouped as 'no comment' for this narrative "
     "classification -- a different convention from cell 8's per-feature diff, stated explicitly.")
print("\n5 examples of a replaced comment:")
print(pd.DataFrame(replaced_examples).to_string(index=False))
comment_summary

A comment counts as 'present' here only when it is non-null and non-blank; null and an empty or whitespace-only string are both grouped as 'no comment' for this narrative classification -- a different convention from cell 8's per-feature diff, stated explicitly.

5 examples of a replaced comment:
   layer          document annotator                                                                                                                         baseline_comment                                annotator_comment
evidence event_2025aji.xmi   Camille Redshift without explicit attribution to event or context; verify whether it belongs to the event/afterglow/host or to a context galaxy.                                 redshift context
evidence event_2025aji.xmi   Camille Redshift without explicit attribution to event or context; verify whether it belongs to the event/afterglow/host or to a context galaxy.          already reported in a previous circular
evidence event_2025aji.xmi   Patr

,layer,identical,annotator_replaced,baseline_none_annotator_wrote,annotator_removed,both_absent
0,evidence,767,133,61,16,3837
1,photometry,509,313,200,60,860


In [9]:
rows = []
for layer, features in FEATURES_BY_LAYER.items():
    for feature in features:
        supplied = [(b, a) for b, a in all_matched_pairs[layer]
                   if (b[feature] is None or b[feature] == "") and a[feature] not in (None, "")]
        if not supplied:
            continue
        examples = [a[feature] for _, a in supplied[:3]] + [None] * 3
        rows.append({"layer": layer, "feature": feature, "count": len(supplied),
                     "example_1": examples[0], "example_2": examples[1], "example_3": examples[2]})

extractor_gaps = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
extractor_gaps

,layer,feature,count,example_1,example_2,example_3
0,photometry,instrument,213,VLT,NOT/ALFOSC,LCO/Sinistro
1,photometry,comment,200,obs time is the mid time,the date was the obs start date. i changed it to the date next to the measurement. Added Instrument,the date was the obs start date. i changed it to the date next to the measurement. Added Instrument
2,evidence,comment,61,Added units of J2000 degrees,"first localization usually not used as we take the skylocalization, we search more candidates within the circle",it is an uncertainty
3,photometry,obs_time_raw,34,Jan 29.237 UT,61201.96762,61201.98177
4,photometry,photometric_band,32,clear,C,i'
5,photometry,exposure_time_raw,25,300s,240s,240s
6,photometry,magnitude_error,11,0.02,0.03,0.02
7,photometry,timezone_raw,6,UT,UT,UT
8,evidence,unit,3,J2000 degrees,primary trigger instrument - first alert,J2000 degrees
9,photometry,limit_sigma,1,10,None,None


In [10]:
def count_overlaps(df):
    """Unordered pairs of spans that overlap within the same (document, layer_source)."""
    total = 0
    for _, group in df.groupby(["document_name", "layer_source"]):
        b, e = group["begin"].to_numpy(), group["end"].to_numpy()
        for i in range(len(group)):
            for j in range(i + 1, len(group)):
                if b[i] < e[j] and b[j] < e[i]:
                    total += 1
    return total


consistency_rows = []
for layer, table in LAYER_TABLES.items():
    consistency_rows.append({
        "layer": layer,
        "overlapping_span_pairs": count_overlaps(table),
        "duplicate_spans": int(table.duplicated(subset=["document_name", "layer_source", "begin", "end"]).sum()),
        "blank_covered_text": int(table["covered_text"].str.strip().eq("").sum()),
        "duplicate_xmi_id": int(table.duplicated(subset=["document_name", "layer_source", "xmi_id"]).sum()),
    })
consistency = pd.DataFrame(consistency_rows)
print("Consistency checks per layer:")
print(consistency.to_string(index=False))

print("\nlabel value counts (evidence_spans):")
print(TABLES["evidence_spans"]["label"].value_counts(dropna=False).to_string())
print("\nmeasurement_type value counts (photometry_spans):")
print(TABLES["photometry_spans"]["measurement_type"].value_counts(dropna=False).to_string())
consistency

Consistency checks per layer:
     layer  overlapping_span_pairs  duplicate_spans  blank_covered_text  duplicate_xmi_id
  evidence                      84                0                   0                 0
photometry                      10                7                   0                 0

label value counts (evidence_spans):
label
EVENT_IDENTITY                   2902
COUNTERPART_ASSOCIATION           677
HIGH_ENERGY_PROPERTY              596
TRIGGER_INSTRUMENT                580
LIGHTCURVE_EVOLUTION              512
LOCALIZATION                      335
TRIGGER_TIME                      293
REDSHIFT_EVENT                    192
SPECTROSCOPY                      153
CLASSIFICATION_INTERPRETATION     145
T90                               101
DURATION_GENERAL                   45
REDSHIFT_CONTEXT                   27
HOST_CONTEXT                       22
None                               18
NEGATIVE_STATEMENT                 12

measurement_type value counts (photometry_spans

,layer,overlapping_span_pairs,duplicate_spans,blank_covered_text,duplicate_xmi_id
0,evidence,84,0,0,0
1,photometry,10,7,0,0


In [11]:
coverage_rows = []
for annotator in sorted(TABLES["annotators"]["annotator"].unique()):
    documents = int(TABLES["annotators"].loc[TABLES["annotators"]["annotator"] == annotator,
                                              "document_name"].nunique())
    evidence_n = int((TABLES["evidence_spans"]["layer_source"] == annotator).sum())
    photometry_n = int((TABLES["photometry_spans"]["layer_source"] == annotator).sum())
    subset = match_summary[match_summary["annotator"] == annotator]
    coverage_rows.append({"annotator": annotator, "documents": documents,
                          "evidence_spans": evidence_n, "photometry_spans": photometry_n,
                          "matched_exact": int(subset["matched_exact"].sum()),
                          "baseline_only": int(subset["baseline_only"].sum()),
                          "annotator_only": int(subset["annotator_only"].sum())})
annotator_coverage = pd.DataFrame(coverage_rows).sort_values("annotator").reset_index(drop=True)
annotator_coverage

,annotator,documents,evidence_spans,photometry_spans,matched_exact,baseline_only,annotator_only
0,Andrii,1,237,266,503,0,0
1,Camille,4,670,214,857,5,27
2,Dahlia,3,577,153,700,0,30
3,Eslam,3,477,316,776,0,17
4,Patrice,3,344,120,461,0,3
5,Priyadarshini,3,614,117,672,0,59
6,Sarah,3,608,438,983,0,63
7,Xinyue,2,359,154,512,0,1
8,Yodgor,3,505,158,636,0,27
9,Zhanat,3,536,135,656,0,15


## Decisions taken from these measurements

These decisions were taken by reading the tables above. Each one names
the finding that motivates it and the number of rows it affects. They
are applied by `scripts/gcn_gold/02_normalise.py`, and their effect is
measured in notebook C.

| # | Decision | Motivating finding | Scope |
|---|---|---|---|
| 1 | A span is keyed by `(document_name, layer_source, begin, end, span_index)`. `xmi_id` is retained but never used to relate rows across layers. | A photometry span identical on every feature carries `xmi_id` 159709 in the baseline and 159669 in the annotator layer. Separately, 7 offset ranges legitimately carry two spans, so offsets alone are not unique either. | both span tables |
| 2 | Baseline and annotator spans are matched on exact `(begin, end)`. No tolerance, no overlap rule. | Summed per (document, annotator) pair, 247 spans fail to match on exact offsets: 5 baseline spans have no counterpart in one specific annotator's own layer and 242 annotator spans have no counterpart in the baseline. All 5 baseline spans were matched by at least one other annotator working the same document — Camille's copy of `event_2025aji.xmi` carries no span at the baseline's LOCALIZATION span [2389:2412] ('uncertainty of 3 arcmin'), which Patrice and Xinyue both kept, but Camille created 10 further LOCALIZATION spans elsewhere in that document rather than moving that span's boundary. None of the 247 unmatched spans, on either side, overlaps anything on the other, which is what makes exact-offset matching sufficient. | 247 unmatched spans |
| 3 | Each annotator span carries `match_status`: `accepted`, `corrected`, `deleted` or `created`. Baseline spans carry `baseline`. | All four cases occur. Across both layers, 6,756 spans match the baseline exactly on offsets, 5 evidence spans were deleted, and 242 were created by annotators. | all span rows |
| 4 | `corrected` is decided by comparing every feature, not the category alone. | `label` differs on 9 of 4,814 matched evidence spans and `measurement_type` on 2 of 1,942. Over the same pairs, `photometry.comment` differs on 29.6%, `instrument` on 11.0% and `photometric_system` on 6.7%. One matched span changed six features while its `measurement_type` stayed the same. | 6,756 matched spans |
| 5 | Corrected spans carry `changed_fields`, the list of features that differ from the baseline. | The correction rate alone does not say what the extraction rules got wrong; the field list does. | rows with `match_status` = `corrected` |
| 6 | Overlapping spans within a layer are retained. No overlap is resolved. | 84 overlapping pairs in evidence, of which 81 are partial crossings rather than nesting, and 60 of the 63 found in annotator layers are inherited unchanged from the baseline. `'long GRB'` labelled CLASSIFICATION_INTERPRETATION and `'GRB 250129A'` labelled EVENT_IDENTITY share the word GRB, and three annotators kept that pair as it was. | 94 pairs |
| 7 | Two spans at identical offsets are retained as distinct rows. | 7 offset ranges carry two photometry spans, each recording a different filter over one line of a table — an i' and a g' measurement at the same characters. Three of the seven already exist in the baseline. | 14 rows |
| 8 | Each span carries `comment_status`: `extractor_guidance`, `annotator_note`, `annotator_removed` or `none`. The comment text is retained unchanged. | Comparing matched spans against the baseline: 1,276 comments are identical to the extractor's guidance, 446 were replaced by the annotator, 261 were written where the baseline had none, and 76 were removed. | both span tables |
| 9 | A span created by an annotator, carrying no category but carrying a comment, is flagged `is_annotator_note`. | 11 of the 14 evidence spans an annotator created without a label, and 36 of the 42 photometry spans created without a `measurement_type`, carry a comment. They are notes addressed to the pipeline rather than annotations: one reads that a photometric measurement was not captured by the pre-annotation, another that photometry is missing at a given timestamp, a third that a search found no detection before a stated time. | 47 rows |
| 10 | Spans without a category are retained and flagged, never dropped. | 18 evidence spans carry no `label` and 44 photometry spans no `measurement_type`, none of them in the baseline. Beyond the 47 notes above, 4 are fully worked measurements — 12 or 13 features populated — whose category was left unset. A span still records that a human marked that text. | 62 rows |
| 11 | Numeric and time features are typed into a companion column, the original value retained beside it. | `limit_sigma` holds `18.3 mag` and `r` alongside numeric values; `timezone_raw` holds `300s` and `120x60` beside `UT`. A null in the typed column records that the value was not numeric rather than that it was absent. | both span tables |
| 12 | Free-text values are retained as written. No instrument name, band or comment is normalised. | These are hand-entered by ten annotators. Any equivalence rule would be right in some cases and wrong in others, and a wrong value is worse than an absent one. | no change |
| 13 | The corpus holds 28 independent validations, not a consensus. `INITIAL_CAS` is retained as a layer of its own. | The project defines a curation workflow but it was never run, so no merged layer exists. Every document carries between two and four annotators, which makes agreement measurable but not resolved. | all rows |
| 14 | Values an annotator used that the baseline never emits are flagged `extractor_vocabulary_gap`. | `obs_time_reference` takes `observation_start` 114 times and `observation_mid` 30 times, neither ever produced by the extractor; `timezone_raw` is empty throughout the baseline and carries `UT` 14 times and `UTC` 10 times in annotator layers. No value appears in the baseline and not in an annotator layer — the extractor's vocabulary is a strict subset of the one in use. | ~190 rows |

Checks that ran and found nothing: duplicate `xmi_id` within a layer,
spans whose covered text is blank, and spans whose offsets fall outside
their document. `non_detection` and `unclear` are declared in the
photometry tagset and appear in no row, baseline or annotator.